# 🎬 Final Hybrid Pipeline — Qwen3-Embedding (LoRA) + BGE-Reranker + Topic Filter

**Архитектура:**

```
Запрос
  │
  ▼
BERT topic classifier  ──→  top-3 топика
  │
  ▼
Qwen3-Embedding-0.6B (LoRA fine-tuned)  ──→  FAISS (top-50 из чанков этих топиков)
                          │
                          ▼
              BGE-Reranker-v2-m3 (cross-encoder)
                          │
                          ▼
                  IoU-дедупликация
                          │
                          ▼
                     Top-5 результатов
```

**Отличия от hybrid_pipe_with_topic_filter:**

| # | Было | Стало |
|---|------|-------|
| 1 | BGE-M3 (pretrained) | **Qwen3-Embedding-0.6B (LoRA fine-tuned)** |
| 2 | CLS pooling | **Last-token pooling** + instruction prompt |

## 0. Установка зависимостей

In [1]:
!pip install -q faiss-cpu FlagEmbedding datasets scikit-learn sentence-transformers peft anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.4 MB/s eta 0:00:00


## 1. Импорты

In [2]:
import json
import pickle
import time
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sentence_transformers import SentenceTransformer
from datasets import Dataset
from sklearn.metrics import accuracy_score
import faiss
import warnings

warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


## 2. Конфигурация

In [3]:
# ── Пути ────────────────────────────────────────────────────────────────────
DATA_ROOT          = Path("/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag")
OUTPUT_SUBMISSION  = Path("submission.csv")

# ── Модели ──────────────────────────────────────────────────────────────────
# Qwen3-Embedding-0.6B fine-tuned (LoRA) — скачивается с HF
QWEN3_HF_REPO     = "dopaminemaximizer/rag_ubiza"
QWEN3_BASE_MODEL   = "Qwen/Qwen3-Embedding-0.6B"
QWEN3_LOCAL_DIR    = "/kaggle/input/models/supertima/qwen3-finetuned/pytorch/default/1/qwen3_embedding_0p6b_finetuned"
BGE_RERANK_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Topic Classifier ───────────────────────────────────────────────────────
BERT_MODEL         = "bert-base-multilingual-cased"
TOPIC_CLASSIFIER_DIR = "/kaggle/working/topic_classifier"
TOPIC_TOP_K        = 5
BERT_EPOCHS        = 12
BERT_BATCH         = 32
BERT_LR            = 3e-5
BERT_MAX_LEN       = 128

# ── Чанкование ──────────────────────────────────────────────────────────────
CHUNK_SCALES = [
    # (30,  5),
    (60, 10),
]

# ── Синтетические чанки ──────────────────────────────────────────────────────
SYNTHETIC_FIELDS = ["question_ru", "question_en", "answer_en"]

# ── Retrieval / Reranking ────────────────────────────────────────────────────
FAISS_CANDIDATES   = 50
FINAL_TOP_K        = 5
IOU_DEDUP_THRESH   = 0.8
IOU_EVAL_THRESH    = 0.5

# ── Батчинг ──────────────────────────────────────────────────────────────────
EMBED_BATCH        = 64
RERANK_BATCH       = 32

# ── Устройство ───────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device:          {DEVICE}")
print(f"Embed model:     {QWEN3_HF_REPO} (base: {QWEN3_BASE_MODEL})")
print(f"Reranker model:  {BGE_RERANK_MODEL}")
print(f"Topic model:     {BERT_MODEL} (top-{TOPIC_TOP_K})")
print(f"Chunk scales:    {CHUNK_SCALES}")
print(f"FAISS candidates:{FAISS_CANDIDATES}  →  top-{FINAL_TOP_K}")

Device:          cuda
Embed model:     dopaminemaximizer/rag_ubiza (base: Qwen/Qwen3-Embedding-0.6B)
Reranker model:  BAAI/bge-reranker-v2-m3
Topic model:     bert-base-multilingual-cased (top-5)
Chunk scales:    [(60, 10)]
FAISS candidates:50  →  top-5


## 3. Загрузка данных

In [4]:
test_df  = pd.read_csv(DATA_ROOT / "test/test.csv")
train_df = pd.read_csv(DATA_ROOT / "train/train_qa.csv")

with open(DATA_ROOT / "transcripts.pkl", "rb") as f:
    transcripts = pickle.load(f)

print(f"Test queries:  {len(test_df):>6}")
print(f"Train rows:    {len(train_df):>6}")
print(f"Transcripts:   {len(transcripts):>6} видео")

Test queries:     812
Train rows:      4466
Transcripts:      436 видео


## 4. Обучение Topic Classifier (BERT)

27 классов, augmentation EN+RU, ~5 эпох. На T4 ~2–3 мин.

In [5]:
# Дедупликация по question_id
questions = train_df.drop_duplicates(subset="question_id")[
    ["question_id", "question_en", "question_ru", "topic"]
].reset_index(drop=True)

topics_list = sorted(questions["topic"].unique())
topic2id = {t: i for i, t in enumerate(topics_list)}
id2topic = {i: t for t, i in topic2id.items()}
NUM_CLASSES = len(topics_list)
questions["label"] = questions["topic"].map(topic2id)

# Augmentation EN + RU
rows_aug = []
for _, row in questions.iterrows():
    rows_aug.append({"text": row["question_en"], "label": row["label"]})
    rows_aug.append({"text": row["question_ru"], "label": row["label"]})

full_dataset = Dataset.from_list(rows_aug)

# Токенизация
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
def tokenize_fn(examples):
    return bert_tokenizer(examples["text"], truncation=True, padding="max_length", max_length=BERT_MAX_LEN)

tokenized = full_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized.set_format("torch")

split = tokenized.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]
print(f"Questions: {len(questions)}, Topics: {NUM_CLASSES}, Train: {len(train_ds)}, Val: {len(val_ds)}")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3784 [00:00<?, ? examples/s]

Questions: 1892, Topics: 27, Train: 3405, Val: 379


In [6]:
def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    top3 = np.argsort(logits, axis=-1)[:, -3:]
    top3_acc = sum(1 for i in range(len(labels)) if labels[i] in top3[i]) / len(labels)
    return {"accuracy": acc, "top3_accuracy": top3_acc}

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=NUM_CLASSES, id2label=id2topic, label2id=topic2id,
)

cls_args = TrainingArguments(
    output_dir=TOPIC_CLASSIFIER_DIR,
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH,
    per_device_eval_batch_size=BERT_BATCH * 2,
    learning_rate=BERT_LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="top3_accuracy",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

cls_trainer = Trainer(
    model=bert_model, args=cls_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_cls_metrics,
)

print("Training topic classifier...")
cls_trainer.train()
print("Done!")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated

Training topic classifier...


Epoch,Training Loss,Validation Loss,Accuracy,Top3 Accuracy
1,2.668216,1.659020,0.781003,0.870712
2,0.788809,0.593019,0.868074,0.952507
3,0.381418,0.379840,0.891821,0.973615
4,0.224613,0.315886,0.918206,0.978892
5,0.118985,0.269257,0.934037,0.981530
6,0.064597,0.219293,0.947230,0.984169
7,0.039293,0.262007,0.936675,0.978892
8,0.022680,0.268737,0.936675,0.981530
9,0.010431,0.251835,0.934037,0.984169
10,0.010544,0.253229,0.944591,0.984169


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Done!


In [7]:
def predict_topics(texts, top_k=TOPIC_TOP_K):
    """Предсказывает top-k топиков для списка текстов."""
    bert_model.eval()
    bert_model.to(DEVICE)
    encodings = bert_tokenizer(texts, truncation=True, padding="max_length",
                               max_length=BERT_MAX_LEN, return_tensors="pt")
    all_top_topics = []
    bs = 64
    for i in range(0, len(texts), bs):
        batch = {k: v[i:i+bs].to(DEVICE) for k, v in encodings.items()}
        with torch.no_grad():
            logits = bert_model(**batch).logits.cpu().numpy()
        top_ids = np.argsort(logits, axis=-1)[:, -top_k:][:, ::-1]
        for row in top_ids:
            all_top_topics.append([id2topic[j] for j in row])
    return all_top_topics

# Quick test
for q, t in zip(
    ["How to cook Italian pasta?", "Как играть на гитаре?"],
    predict_topics(["How to cook Italian pasta?", "Как играть на гитаре?"])
):
    print(f"  {q} → {t}")

  How to cook Italian pasta? → ['travel-italy', 'cooking-tutorials', 'sing-opera', 'DIY', 'travel-usa']
  Как играть на гитаре? → ['playing-drums', 'learn-vocals', 'cooking-tutorials', 'playing-guitar', 'playing-piano']


## 5. Построение чанков (multi-scale + synthetic + topic)

In [8]:
def extract_hash(path_str: str) -> str:
    stem = Path(path_str).stem
    parts = stem.split("_", 1)
    return parts[1] if len(parts) > 1 else stem

hash_to_tr_key = {extract_hash(k): k for k in transcripts.keys()}

# Маппинг video_hash → topic
video_hash_to_topic = {}
for _, row in train_df.iterrows():
    h = extract_hash(row["video_file"])
    video_hash_to_topic[h] = row["topic"]

def build_transcript_chunks(transcripts, chunk_sec, overlap_sec):
    out = []
    stride = chunk_sec - overlap_sec
    for video_file, segments in transcripts.items():
        if not segments:
            continue
        h = extract_hash(video_file)
        topic = video_hash_to_topic.get(h, "unknown")
        total_end = segments[-1]["end"]
        start = 0.0
        while start < total_end - 1:
            end = min(start + chunk_sec, total_end)
            texts = [seg["text"] for seg in segments if seg["start"] < end and seg["end"] > start]
            if texts:
                out.append({
                    "video_file": video_file, "start": start, "end": end,
                    "text": " ".join(texts),
                    "source": f"transcript_{int(chunk_sec)}s",
                    "topic": topic,
                })
            start += stride
    return out

def build_synthetic_chunks(train_df, hash_to_tr_key, fields):
    out = []
    available = [f for f in fields if f in train_df.columns]
    for _, row in train_df.iterrows():
        video_hash = extract_hash(row["video_file"])
        tr_key = hash_to_tr_key.get(video_hash, row["video_file"])
        for field in available:
            text = str(row[field]).strip()
            if not text or text == "nan":
                continue
            out.append({
                "video_file": tr_key, "start": float(row["start"]), "end": float(row["end"]),
                "text": text,
                "source": f"synthetic_{field}",
                "topic": row["topic"],
            })
    return out

# Build
tr_chunks = []
for chunk_sec, overlap_sec in CHUNK_SCALES:
    scale = build_transcript_chunks(transcripts, chunk_sec, overlap_sec)
    print(f"  Transcript chunks ({chunk_sec}s, overlap={overlap_sec}s): {len(scale)}")
    tr_chunks.extend(scale)

syn_chunks = build_synthetic_chunks(train_df, hash_to_tr_key, SYNTHETIC_FIELDS)
chunks = tr_chunks + syn_chunks

print(f"\n  Transcript: {len(tr_chunks)}, Synthetic: {len(syn_chunks)}, Total: {len(chunks)}")

# Индекс чанков по топику
topic_to_chunk_ids = defaultdict(list)
for i, c in enumerate(chunks):
    topic_to_chunk_ids[c["topic"]].append(i)
print(f"  Topics with chunks: {len(topic_to_chunk_ids)}")

  Transcript chunks (60s, overlap=10s): 6557

  Transcript: 6557, Synthetic: 13398, Total: 19955
  Topics with chunks: 28


## 6. Qwen3-Embedding-0.6B (LoRA fine-tuned) + FAISS

In [12]:
print(f'Загружаем Qwen3 fine-tuned из {QWEN3_LOCAL_DIR} ...')
embed_model = SentenceTransformer(str(QWEN3_LOCAL_DIR), device=DEVICE)
print(f'Max seq length: {embed_model.max_seq_length}')
print(f'Embedding dim:  {embed_model.get_sentence_embedding_dimension()}')

Загружаем Qwen3 fine-tuned из /kaggle/input/models/supertima/qwen3-finetuned/pytorch/default/1/qwen3_embedding_0p6b_finetuned ...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights: 0it [00:00, ?it/s]

Qwen3Model LOAD REPORT from: /kaggle/input/models/supertima/qwen3-finetuned/pytorch/default/1/qwen3_embedding_0p6b_finetuned
Key                                                                     | Status     | 
------------------------------------------------------------------------+------------+-
base_model.model.layers.{0...27}.self_attn.o_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.v_proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.k_proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.q_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.q_proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.v_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.k_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.layers.{0...27}.self_attn.o_proj.lora_B.default.weight | UNEXPECTE

Max seq length: 512
Embedding dim:  1024


In [13]:

# --------------------------------------------------
# 6. Prompt (как при обучении)
# --------------------------------------------------

QUERY_INSTRUCTION = (
    "Instruct: Given a question about a video transcript, "
    "retrieve the most relevant transcript passage.\nQuery: "
)


# --------------------------------------------------
# 7. Encoding functions
# --------------------------------------------------

def embed_texts_document(texts, batch_size=EMBED_BATCH):
    return embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)


def embed_texts_query(texts, batch_size=EMBED_BATCH):

    prompted = [QUERY_INSTRUCTION + t for t in texts]

    return embed_model.encode(
        prompted,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)


# --------------------------------------------------
# 8. Quick test
# --------------------------------------------------

q = "How to cook pasta?"

e = embed_texts_query([q])

print("Embedding shape:", e.shape)
print("Embedding dim:", embed_model.get_sentence_embedding_dimension())

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (1, 1024)
Embedding dim: 1024


In [14]:
from openai import OpenAI
import os
from concurrent.futures import ThreadPoolExecutor

# ── Конфигурация HyDE ─────────────────────────────────────────────────────────
HYDE_ENABLED    = True   # False → pipeline без изменений
HYDE_ALPHA      = 0.3    # 0.0 = чистый HyDE, 1.0 = оригинал, 0.3 = 70% HyDE
HYDE_MODEL      = "google/gemini-2.5-flash"  # OpenRouter model ID
HYDE_MAX_TOKENS = 180
HYDE_WORKERS    = 16     # параллельные API-запросы (Gemini держит высокий RPS)
HYDE_FEW_SHOT_N = 5      # сколько примеров из train давать в промпт

api_key = env.OPENROUTER_API_KEY
hyde_client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)
print(f"HyDE enabled={HYDE_ENABLED}, alpha={HYDE_ALPHA}, model={HYDE_MODEL}")

# ── Few-shot примеры из train (question_en → answer_en) ──────────────────────
# answer_en — это уже гипотетические ответы в стиле транскриптов, идеальный few-shot
_few_shot_df = (
    train_df[["question_en", "answer_en"]]
    .dropna(subset=["question_en", "answer_en"])
    .drop_duplicates("question_en")
    .sample(n=min(HYDE_FEW_SHOT_N * 3, len(train_df)), random_state=42)
    .head(HYDE_FEW_SHOT_N)
)

_FEW_SHOT_BLOCK = "\n\n".join(
    f"Question: {row.question_en}\nTranscript fragment: {row.answer_en}"
    for row in _few_shot_df.itertuples()
)

_SYSTEM_PROMPT = (
    "You generate short transcript fragments from educational video lectures. "
    "The fragments must sound like natural spoken English — as if a teacher or coach "
    "is speaking to the camera. 2–4 sentences, no formal headings, no lists."
)

def _build_hyde_user_prompt(question: str) -> str:
    return (
        f"Here are examples of questions and matching transcript fragments:\n\n"
        f"{_FEW_SHOT_BLOCK}\n\n"
        f"Now write a transcript fragment that answers this question:\n"
        f"Question: {question}\nTranscript fragment:"
    )


def _generate_one(question: str) -> str:
    try:
        resp = hyde_client.chat.completions.create(
            model=HYDE_MODEL,
            max_tokens=HYDE_MAX_TOKENS,
            messages=[
                {"role": "system", "content": _SYSTEM_PROMPT},
                {"role": "user", "content": _build_hyde_user_prompt(question)},
            ],
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"[HyDE] API error for '{question[:40]}...': {e}")
        return question  # fallback: оригинальный вопрос


def generate_hypothetical_docs(questions: list[str]) -> list[str]:
    """Параллельная генерация гипотетических транскриптов для списка вопросов."""
    with ThreadPoolExecutor(max_workers=HYDE_WORKERS) as pool:
        results = list(tqdm(
            pool.map(_generate_one, questions),
            total=len(questions),
            desc="HyDE generation",
        ))
    return results


def embed_with_hyde(questions: list[str], alpha: float = HYDE_ALPHA):
    """
    Возвращает L2-нормализованные эмбеддинги (numpy float32):
        alpha=0.0 → embed(hypothetical_doc)   — чистый HyDE
        alpha=1.0 → embed(question)            — без HyDE
        alpha=0.3 → 70% HyDE + 30% query      — рекомендуемое
    Также возвращает список сгенерированных гипотетических документов.
    """
    query_embs = embed_texts_query(questions, batch_size=EMBED_BATCH)

    if not HYDE_ENABLED or alpha >= 1.0:
        return query_embs, [""] * len(questions)

    hyp_docs = generate_hypothetical_docs(questions)

    # Гипотетические документы кодируем как документы (без instruction prefix)
    hyp_embs = embed_texts_document(hyp_docs, batch_size=EMBED_BATCH)

    blended = alpha * query_embs + (1.0 - alpha) * hyp_embs
    norms = np.linalg.norm(blended, axis=1, keepdims=True)
    blended = blended / np.maximum(norms, 1e-8)

    return blended.astype(np.float32), hyp_docs


# ── Проверка на нескольких примерах ──────────────────────────────────────────
_sample_qs = [
    "How can strategic pauses improve the effectiveness of a speech?",
    "What is the best way to practise Italian pronunciation?",
]
_hyp_docs_sample = generate_hypothetical_docs(_sample_qs)
for q, h in zip(_sample_qs, _hyp_docs_sample):
    print(f"\nQ: {q}\nH: {h}")

HyDE enabled=True, alpha=0.3, model=google/gemini-2.5-flash


HyDE generation:   0%|          | 0/2 [00:00<?, ?it/s]


Q: How can strategic pauses improve the effectiveness of a speech?
H: Strategic pauses can create a powerful impact. They give your audience a moment to process what you’ve said and anticipate what’s coming next, which really amplifies your message. Also, a well-placed pause helps build suspense and emphasize key points, making your speech much more engaging.

Q: What is the best way to practise Italian pronunciation?
H: Here's a good way to practice your Italian pronunciation. Start by listening to native speakers, really paying attention to the rhythm and intonation, not just the individual sounds. Then, try to mimic what you hear as closely as possible, even if it feels a bit exaggerated at first. Recording yourself and comparing it to the original can be incredibly helpful for spotting where you need to adjust.


## 6.5 HyDE — Hypothetical Document Embedding

**Идея:** вместо `embed(question)` → FAISS генерируем гипотетический фрагмент транскрипта и ищем `embed(hypothetical_transcript)` → FAISS.

Это закрывает семантический gap между короткими вопросами и разговорной речью в транскриптах.

Few-shot примеры берём прямо из `answer_en` в train — это уже готовые образцы нужного стиля.

```
query → LLM (few-shot из answer_en) → hypothetical_transcript
                                              │
                    α * embed_query + (1-α) * embed_hypothetical  →  FAISS
```

In [15]:
chunk_texts = [c["text"] for c in chunks]
t0 = time.time()
chunk_embeddings = embed_texts_document(chunk_texts, batch_size=EMBED_BATCH)
print(f"Chunk embeddings: {chunk_embeddings.shape}, took {time.time()-t0:.0f}s")

# Глобальный FAISS (для fallback)
dim = chunk_embeddings.shape[1]
global_index = faiss.IndexFlatIP(dim)
global_index.add(chunk_embeddings)
print(f"Global FAISS: dim={dim}, vectors={global_index.ntotal:,}")

Batches:   0%|          | 0/312 [00:00<?, ?it/s]

Chunk embeddings: (19955, 1024), took 573s
Global FAISS: dim=1024, vectors=19,955


## 7. BGE-Reranker + Topic-filtered search

In [16]:
class BGEReranker:
    def __init__(self, model_name, device, use_fp16=True):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        if use_fp16 and device == "cuda":
            self.model = self.model.half()
        self.model = self.model.to(device).eval()
        self.device = device

    @torch.no_grad()
    def compute_score(self, pairs, batch_size=32, normalize=True):
        all_scores = []
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            queries, passages = zip(*batch)
            enc = self.tok(list(queries), list(passages), padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(self.device)
            logits = self.model(**enc).logits.squeeze(-1).float().cpu().tolist()
            all_scores.extend(logits if isinstance(logits, list) else [logits])
        if normalize:
            all_scores = [1.0 / (1.0 + np.exp(-s)) for s in all_scores]
        return all_scores

print(f"Загружаем {BGE_RERANK_MODEL} ...")
reranker = BGEReranker(BGE_RERANK_MODEL, device=DEVICE, use_fp16=True)
print("Reranker загружен.")

Загружаем BAAI/bge-reranker-v2-m3 ...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker загружен.


## 8. Retrieval pipeline: topic filter → FAISS → rerank → dedup

In [17]:
def compute_iou(a_start, a_end, b_start, b_end):
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return inter / union if union > 0 else 0.0


def dedup_by_iou(candidates, top_k=5, iou_threshold=0.8):
    selected = []
    for cand in candidates:
        is_dup = False
        for sel in selected:
            if extract_hash(cand["video_file"]) == extract_hash(sel["video_file"]):
                if compute_iou(cand["start"], cand["end"], sel["start"], sel["end"]) >= iou_threshold:
                    is_dup = True
                    break
        if not is_dup:
            selected.append(cand)
        if len(selected) == top_k:
            break
    if len(selected) < top_k:
        for cand in candidates:
            if cand not in selected:
                selected.append(cand)
            if len(selected) == top_k:
                break
    return selected[:top_k]


def topic_filtered_faiss_search(query_embs, query_topics_list, n_candidates=FAISS_CANDIDATES):
    """
    Для каждого запроса: собираем чанки из предсказанных топиков,
    строим мини-FAISS, ищем top-n_candidates.
    Возвращает list[list[dict]] — кандидаты для каждого запроса.
    """
    all_candidates = []
    for i in range(len(query_embs)):
        q_emb = query_embs[i:i+1].astype(np.float32)
        pred_topics = query_topics_list[i]

        # Собираем глобальные ID чанков из предсказанных топиков
        candidate_ids = []
        for t in pred_topics:
            if t in topic_to_chunk_ids:
                candidate_ids.extend(topic_to_chunk_ids[t])
        candidate_ids = list(set(candidate_ids))

        # Fallback на глобальный если слишком мало
        if len(candidate_ids) < n_candidates:
            scores, ids = global_index.search(q_emb, n_candidates)
            cands = [
                {**chunks[idx], "faiss_score": float(sc)}
                for idx, sc in zip(ids[0], scores[0]) if idx >= 0
            ]
            all_candidates.append(cands)
            continue

        # Мини-FAISS
        cand_embs = chunk_embeddings[candidate_ids].astype(np.float32)
        mini_idx = faiss.IndexFlatIP(dim)
        mini_idx.add(cand_embs)
        scores, local_ids = mini_idx.search(q_emb, min(n_candidates, mini_idx.ntotal))

        cands = []
        for li, sc in zip(local_ids[0], scores[0]):
            if li >= 0:
                gi = candidate_ids[li]
                cands.append({**chunks[gi], "faiss_score": float(sc)})
        all_candidates.append(cands)

    return all_candidates

print("Pipeline functions ready")

Pipeline functions ready


## 9. Eval на train

## 10. Inference → submission.csv

In [18]:
test_queries = test_df["question"].tolist()

# ── 1. HyDE: генерируем гипотетические документы и кодируем ─────────────────
print(f"Кодируем test-запросы (HyDE={HYDE_ENABLED}, alpha={HYDE_ALPHA}) ...")
test_query_embs, test_hyp_docs = embed_with_hyde(test_queries, alpha=HYDE_ALPHA)

# Сохраняем гипотетические документы для отладки
hyp_docs_df = pd.DataFrame({"question": test_queries, "hypothetical_doc": test_hyp_docs})
hyp_docs_df.to_csv("hyde_docs_test.csv", index=False)
print(f"Гипотетические документы сохранены → hyde_docs_test.csv")

# ── 2. Topic prediction ───────────────────────────────────────────────────────
print("Предсказываем топики ...")
test_topics_pred = predict_topics(test_queries, top_k=TOPIC_TOP_K)

# ── 3. Topic-filtered FAISS ──────────────────────────────────────────────────
print("Topic-filtered FAISS search ...")
test_candidates = topic_filtered_faiss_search(test_query_embs, test_topics_pred)

# ── 4. Rerank + dedup + submission ───────────────────────────────────────────
submission_rows = []
for qid_idx, qid in enumerate(tqdm(test_df["query_id"], desc="Building submission")):
    query_text = test_queries[qid_idx]
    candidates = test_candidates[qid_idx]

    if candidates:
        pairs = [(query_text, c["text"]) for c in candidates]
        rerank_scores = reranker.compute_score(pairs, normalize=True)
        for c, s in zip(candidates, rerank_scores):
            c["rerank_score"] = float(s)
        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    top5 = dedup_by_iou(candidates, top_k=FINAL_TOP_K, iou_threshold=IOU_DEDUP_THRESH)

    row = {"query_id": qid}
    for rank in range(1, FINAL_TOP_K + 1):
        row[f"video_file_{rank}"] = ""
        row[f"start_{rank}"] = 0.0
        row[f"end_{rank}"] = 0.0
    for rank, c in enumerate(top5, start=1):
        row[f"video_file_{rank}"] = Path(c["video_file"]).stem
        row[f"start_{rank}"] = c["start"]
        row[f"end_{rank}"] = c["end"]
    submission_rows.append(row)

cols = ["query_id"]
for rank in range(1, FINAL_TOP_K + 1):
    cols.extend([f"video_file_{rank}", f"start_{rank}", f"end_{rank}"])
submission = pd.DataFrame(submission_rows)[cols]
submission.to_csv(OUTPUT_SUBMISSION, index=False, encoding="utf-8")

print(f"\nSubmission saved: {OUTPUT_SUBMISSION}")
print(f"Shape: {submission.shape}")
submission.head(3)

Кодируем test-запросы (HyDE=True, alpha=0.3) ...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

HyDE generation:   0%|          | 0/812 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Гипотетические документы сохранены → hyde_docs_test.csv
Предсказываем топики ...
Topic-filtered FAISS search ...


Building submission:   0%|          | 0/812 [00:00<?, ?it/s]


Submission saved: submission.csv
Shape: (812, 16)


,query_id,video_file_1,start_1,end_1,video_file_2,start_2,end_2,video_file_3,start_3,end_3,video_file_4,start_4,end_4,video_file_5,start_5,end_5
0,1102,video_0fba03d9,350.0,410.0,video_0fba03d9,400.0,460.0,video_0fba03d9,0.0,60.0,video_0fba03d9,50.0,110.0,video_0fba03d9,41.0,105.0
1,530,video_4f2ceb30,202.0,267.0,video_7365243c,440.0,534.0,video_10e08ef3,247.0,319.0,video_6d8e8e79,400.0,460.0,video_91b73ab3,80.0,130.0
2,2698,video_4d49b209,287.0,365.0,video_4a4de4ba,676.0,715.0,video_4a4de4ba,526.0,603.0,video_b064e921,116.0,148.0,video_4d49b209,287.0,365.0


In [19]:
print("Кодируем test-запросы ...")
test_queries = test_df["question"].tolist()
test_query_embs = embed_texts_query(test_queries, batch_size=EMBED_BATCH)

print("Предсказываем топики ...")
test_topics_pred = predict_topics(test_queries, top_k=TOPIC_TOP_K)

print("Topic-filtered FAISS search ...")
test_candidates = topic_filtered_faiss_search(test_query_embs, test_topics_pred)

# Rerank + dedup + submission
submission_rows = []
for qid_idx, qid in enumerate(tqdm(test_df["query_id"], desc="Building submission")):
    query_text = test_queries[qid_idx]
    candidates = test_candidates[qid_idx]

    if candidates:
        pairs = [(query_text, c["text"]) for c in candidates]
        rerank_scores = reranker.compute_score(pairs, normalize=True)
        for c, s in zip(candidates, rerank_scores):
            c["rerank_score"] = float(s)
        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)

    top5 = dedup_by_iou(candidates, top_k=FINAL_TOP_K, iou_threshold=IOU_DEDUP_THRESH)

    row = {"query_id": qid}
    for rank in range(1, FINAL_TOP_K + 1):
        row[f"video_file_{rank}"] = ""
        row[f"start_{rank}"] = 0.0
        row[f"end_{rank}"] = 0.0
    for rank, c in enumerate(top5, start=1):
        row[f"video_file_{rank}"] = Path(c["video_file"]).stem
        row[f"start_{rank}"] = c["start"]
        row[f"end_{rank}"] = c["end"]
    submission_rows.append(row)

cols = ["query_id"]
for rank in range(1, FINAL_TOP_K + 1):
    cols.extend([f"video_file_{rank}", f"start_{rank}", f"end_{rank}"])
submission = pd.DataFrame(submission_rows)[cols]
submission.to_csv(OUTPUT_SUBMISSION, index=False, encoding="utf-8")

print(f"\nSubmission saved: {OUTPUT_SUBMISSION}")
print(f"Shape: {submission.shape}")
submission.head(3)

Кодируем test-запросы ...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Предсказываем топики ...
Topic-filtered FAISS search ...


Building submission:   0%|          | 0/812 [00:00<?, ?it/s]


Submission saved: submission.csv
Shape: (812, 16)


,query_id,video_file_1,start_1,end_1,video_file_2,start_2,end_2,video_file_3,start_3,end_3,video_file_4,start_4,end_4,video_file_5,start_5,end_5
0,1102,video_0fba03d9,350.0,410.0,video_0fba03d9,400.0,460.0,video_0fba03d9,0.0,60.0,video_0fba03d9,50.0,110.0,video_0fba03d9,41.0,105.0
1,530,video_4f2ceb30,202.0,267.0,video_7365243c,440.0,534.0,video_10e08ef3,247.0,319.0,video_6d8e8e79,400.0,460.0,video_91b73ab3,80.0,130.0
2,2698,video_4d49b209,287.0,365.0,video_4a4de4ba,676.0,715.0,video_4a4de4ba,526.0,603.0,video_9bd1c35d,0.0,230.0,video_b064e921,319.0,337.0


In [20]:
!zip -r -j /kaggle/working/topic_classifier.zip /kaggle/working/topic_classifier 

  adding: model.safetensors (deflated 7%)
  adding: optimizer.pt (deflated 53%)
  adding: scaler.pt (deflated 64%)
  adding: trainer_state.json (deflated 72%)
  adding: rng_state.pth (deflated 26%)
  adding: training_args.bin (deflated 53%)
  adding: scheduler.pt (deflated 61%)
  adding: config.json (deflated 63%)
